<a id="top"></a>

# L50 · Agent mini-project — end-to-end walkthrough

```yaml
title: "L50 · Agent mini-project — end-to-end walkthrough"
keywords: capstone, mini-project, vertical slice, tool design, shallow agent, create_agent, agent_graph, trace, TraceEvent, Langfuse, eval case, scorer, regression, receipt reconciliation, expense policy, LLM-in-the-loop tool
estimated duration: 70
```

> **Lesson:** L50 — Agent mini-project (mini-track capstone). **This is a walkthrough, not a lecture+lab:** the proctor drives from a blank cell to a traced, evaluated shallow agent, and you rebuild it alongside, cell by cell.
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L50/objectives.md) · [demos_or_activities.md](../../../../docs/origin/lesson_roadmaps/L50/demos_or_activities.md).
> **Read first:** [L5001_intro](L5001_intro.md), then the framing deck [L5002_lecture](L5002_lecture.md). Read [MINI_WRAPUP](../MINI_WRAPUP.md) *after* this.
> **Anchor model: Claude Sonnet 4.6.** The walkthrough runs on scripted stand-ins (`FakeModel` for the agent's brain, a scripted policy judge for `check_expense_policy`) so it's reproducible offline. The **live** cells — a real Sonnet policy read (Section 2) and the Langfuse trace push (Section 4) — are clearly gated and light up when the cohort stack is configured.

**Nothing here is new — that's the point.** Every piece you'll touch — the loop, the trace, the eval types — you already built in an earlier lesson, and it all lives in `fluffy_potato_curriculum.common`. The *one* thing you author fresh is a single small tool. L50 is those five skills becoming **one build** on a task you scoped.

## Contents

- [0. Setup](#0-setup)
- [1. Scope the vertical slice](#1-scope-the-vertical-slice) — one task, one tool, a "done" line, and the non-goals
- [2. Design the one new tool](#2-design-the-one-new-tool) — the only fresh code: `find_matching_record`
- [3. Assemble the shallow agent](#3-assemble-the-shallow-agent) — `agent_graph.run`, the L10 loop packaged
- [4. Trace it and find a real failure](#4-trace-it-and-find-a-real-failure) — read the trace, name the failure
- [5. Turn the failure into one eval case](#5-turn-the-failure-into-one-eval-case) — one case, one scorer, one run
- [6. Reflect and hand off](#6-reflect-and-hand-off) — the finished slice, and what you'd add next

## 0. Setup

One setup cell: the shared `common/` layer you'll assemble from, the offline data bundle, and two small helpers. Run it once, top to bottom.

In [1]:
import json
from pathlib import Path
from typing import Any

import fluffy_potato_curriculum
from fluffy_potato_curriculum.common.agent_graph import RunResult, arun
from fluffy_potato_curriculum.common.config import get_settings, langfuse_configured
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    tool_calls,
    tool_trajectory,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import calculator

# The receipt-reconciliation data bundle that ships with this lesson. We locate it via
# the installed package (not the notebook's cwd) so it loads no matter where Jupyter runs.
_DATA_DIR = Path(fluffy_potato_curriculum.__file__).parent / "lessons" / "L50" / "data"
RECORDS_LEDGER: list[dict[str, Any]] = json.loads((_DATA_DIR / "records_ledger.json").read_text())
RECEIPTS: list[dict[str, Any]] = json.loads((_DATA_DIR / "receipts.json").read_text())


def receipt_text(receipt_id: str) -> str:
    """Return a receipt's raw payload as the string an agent would be handed."""
    receipt = next(r for r in RECEIPTS if r["id"] == receipt_id)
    raw = receipt["raw"]
    return raw if isinstance(raw, str) else json.dumps(raw)


def narrate(result: RunResult) -> None:
    """Print one line per span, in run order -- the trace, read top to bottom (the L12 move)."""
    for i, span in enumerate(result.trace):
        print(f"[{i}] {span.one_line()}")


# Live cells (Section 4) are gated on these so a keyless restart-and-run-all still passes.
LIVE_LLM = bool(get_settings().anthropic_api_key)
LIVE_LANGFUSE = langfuse_configured()
print("records:", len(RECORDS_LEDGER), "| receipts:", len(RECEIPTS))
print("live model:", LIVE_LLM, "| live Langfuse:", LIVE_LANGFUSE)

/Users/timlee/myrepos/fluffy-potato-curriculum/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


records: 4 | receipts: 5
live model: False | live Langfuse: False


## 1. Scope the vertical slice

A capstone is a **vertical slice**, not a product: one narrow path that goes *all the way through* the stack — goal → tool → agent → trace → eval — rather than a wide feature that only reaches halfway. Your job in this segment is restraint: cut a fuzzy goal down to the thinnest thing that still exercises all five objectives, then **stop**.

The goal in one line: *an agent that reconciles a receipt against our expense records.* Now cut it down — and write the **non-goals** in plain sight, because a mini-project is defined as much by what it refuses to do as by what it does.

In [2]:
# The whole spec on one screen. Naming the non-goals *is* the scoping skill.
SPEC = {
    "one_task": "Given one receipt, find its matching record, check it reconciles, and flag a policy breach.",
    "one_new_tool": "find_matching_record(receipt) -> the matched record, or 'no confident match'  (the only tool you write).",
    "provided_tools": "calculator (reused from common) + check_expense_policy (pre-built) -- the agent runs them; you don't author them.",
    "done_when": "the agent answers, the run is traced, and one eval case would catch its regression.",
    "non_goals": [
        "no live bank / accounting API -- the data is a small offline bundle",
        "no multi-receipt batches -- one receipt at a time",
        "no reimbursement detection -- that's the 'what you'd add next' (Section 6)",
        "no planning / memory / subagents -- a shallow agent is plenty",
    ],
}
for line in SPEC["non_goals"]:
    print("NOT today:", line)
print("\nDone when:", SPEC["done_when"])

NOT today: no live bank / accounting API -- the data is a small offline bundle
NOT today: no multi-receipt batches -- one receipt at a time
NOT today: no reimbursement detection -- that's the 'what you'd add next' (Section 6)
NOT today: no planning / memory / subagents -- a shallow agent is plenty

Done when: the agent answers, the run is traced, and one eval case would catch its regression.


Apply the L08 *is-a-tool-warranted?* test: could the model reconcile a receipt reliably in its head? No — a receipt arrives in a different shape from every source (a cafe's JSON, a rideshare's differently-named fields, a hotel's raw text), and cross-format normalize-and-match is exactly the kind of fiddly, deterministic work you do **not** want a model doing by eyeball. So the one tool is genuinely justified — which means there's something real to trace and eval.

[↑ Back to top](#top)

## 2. Design the one new tool

This is the *only* tool you author from scratch in the whole lesson — so it's the one place the L07/L08 tool-design checklist gets a fresh workout: a plain typed function, a docstring that reads as its contract, and **error handling that turns a bad input into a value, never a crash.** Everything else the agent runs is imported.

First, look at what makes this tool *warranted* — the same expense, three completely different receipt shapes:

In [3]:
for rid in ("R-coffee", "R-ride", "R-hotel"):
    print(f"--- {rid} ---")
    print(receipt_text(rid))
    print()

--- R-coffee ---
{"merchant": "Blue Bottle Coffee", "date": "2026-06-03", "line_items": [{"name": "Latte", "price": 5.5}, {"name": "Drip coffee", "price": 3.25}, {"name": "Almond croissant", "price": 4.0}], "total": 12.75}

--- R-ride ---
{"company": "Citywide Taxi", "trip_date": "2026-06-05", "amount_charged": 41.2, "distance_mi": 8.3}

--- R-hotel ---
Harbor Grand Hotel
Folio #88231
Check-in: 2026-06-06   Check-out: 2026-06-07
Room charge: 240.00
City tax: 28.40
Amount due: 268.40



The vendor is `merchant` in one, `company` in another, and the first line of a text blob in the third. Normalizing across that is the tool's real job. Here it is — the same code as the tested sibling `receipt_tools.py`, written live:

In [4]:
import re
from dataclasses import dataclass

# The same field wears a different name per source -- this alias table IS the normalization.
_VENDOR_KEYS = ("merchant", "vendor", "company", "store", "name")
_AMOUNT_KEYS = ("total", "amount", "amount_due", "amount_charged", "grand_total")
_AMOUNT_TOLERANCE = 0.01
_TEXT_AMOUNT_RE = re.compile(
    r"(?:amount due|total|amount|balance)\s*[:$]*\s*\$?\s*([0-9]+(?:\.[0-9]{1,2})?)", re.IGNORECASE
)


@dataclass(frozen=True)
class Normalized:
    vendor: str
    total: float


def _coerce_amount(value: Any) -> float | None:
    """Parse a number from int/float/str, or None -- this is where 'amt: ??' becomes a value."""
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value.strip().lstrip("$"))
        except ValueError:
            return None
    return None


def _normalize(receipt: str) -> Normalized | None:
    """Coerce any supported format (JSON dict, or a raw folio text blob) to (vendor, total)."""
    try:
        parsed = json.loads(receipt)
    except json.JSONDecodeError:
        parsed = None
    if isinstance(parsed, dict):  # cafe / rideshare / scan
        vendor = next((parsed[k] for k in _VENDOR_KEYS if k in parsed), None)
        total = _coerce_amount(next((parsed[k] for k in _AMOUNT_KEYS if k in parsed), None))
    else:  # hotel folio: raw text -- vendor is the first line, amount is by label
        lines = [ln.strip() for ln in receipt.splitlines() if ln.strip()]
        match = _TEXT_AMOUNT_RE.search(receipt)
        vendor = lines[0] if lines else None
        total = _coerce_amount(match.group(1)) if match else None
    if not isinstance(vendor, str) or not vendor.strip() or total is None:
        return None
    return Normalized(vendor=vendor, total=total)


def find_matching_record(receipt: str) -> str:
    """Match one receipt against the expense records ledger; return the record or a miss.

    Pass the receipt exactly as you received it -- a JSON object as text (cafe, rideshare)
    or the raw printed lines (a hotel folio). Returns a JSON string: either
    {"match": {...record...}} on a confident match, or {"match": null, "reason": ...}.
    It never raises: an unreadable receipt resolves to "match": null so the agent can
    report "no confident match" gracefully instead of looping on a bad input.
    """
    norm = _normalize(receipt)
    if norm is None:
        return json.dumps({"match": None, "reason": "unrecognized receipt format"})
    wanted = " ".join(norm.vendor.lower().split())
    hits = [
        rec
        for rec in RECORDS_LEDGER
        if " ".join(str(rec["vendor"]).lower().split()) == wanted
        and abs(float(rec["amount"]) - norm.total) <= _AMOUNT_TOLERANCE
    ]
    if len(hits) == 1:
        return json.dumps({"match": hits[0]})
    return json.dumps({"match": None, "reason": "no confident match in the ledger"})

In [5]:
# Smoke-test the tool the way L08 says to: the happy formats match, and the malformed
# one fails GRACEFULLY (a value, not an exception).
for rid in ("R-coffee", "R-ride", "R-hotel", "R-unknown-vendor", "R-mystery"):
    print(f"{rid:18}", find_matching_record(receipt_text(rid)))

R-coffee           {"match": {"record_id": "EXP-1001", "vendor": "blue bottle coffee", "amount": 12.75, "date": "2026-06-03", "category": "meals"}}
R-ride             {"match": {"record_id": "EXP-1002", "vendor": "citywide taxi", "amount": 41.2, "date": "2026-06-05", "category": "travel"}}
R-hotel            {"match": {"record_id": "EXP-1003", "vendor": "harbor grand hotel", "amount": 268.4, "date": "2026-06-07", "category": "lodging"}}
R-unknown-vendor   {"match": null, "reason": "no confident match in the ledger"}
R-mystery          {"match": null, "reason": "unrecognized receipt format"}


### The second core tool — provided, not authored (and it's an *LLM read*)

`find_matching_record` is the tool you *write*. But the agent also has to answer *"is this expense even allowed?"* — and company policy isn't a neat cap table; it's **free-form prose** ([`expense_policy.md`](data/expense_policy.md)): paragraphs of guidance with judgment calls ("team meals may exceed $50 *with a manager's note*"). Same L08 test — a model shouldn't recall *your company's* policy from memory — same answer: warranted.

But its *shape* is the opposite of the deterministic matcher. Instead of a lookup, this tool does a **single LLM read**: it hands the policy prose to a model, which interprets it, judges the expense, and **cites the rule it applied**. That's deliberately the "expensive" call where a dict would do — because real policy is guidance to interpret, not a table. It's the one **LLM-in-the-loop tool** in the slice, handed to you **pre-built** in [`receipt_tools.py`](receipt_tools.py) (the authoring budget stays one tool).

Offline it consults a **scripted policy judge** so this notebook stays reproducible and keyless — the same live/offline seam `FakeModel` gives the agent's brain. Import it, read the verdict *and its citation*, and (in class, with a key) watch a real Sonnet read the prose:

In [6]:
from fluffy_potato_curriculum.lessons.L50.receipt_tools import (
    check_expense_policy,
    make_check_expense_policy,
)

# The matched record carries `category` and `amount`, so the agent has what this needs.
# Offline, `check_expense_policy` consults a scripted policy judge (reproducible, keyless):
# meals 12.75 is within the $50 guidance; lodging 268.40 is over the $250 guidance -> flagged.
for category, amount in (("meals", 12.75), ("lodging", 268.40)):
    print(f"{category:9}{amount:>8}", check_expense_policy(category, amount))

# In class (ANTHROPIC_API_KEY set), wire the SAME tool to live Sonnet: now a real model
# reads expense_policy.md and cites the rule -- the "inefficient" LLM read, for real.
if LIVE_LLM:
    from langchain.chat_models import init_chat_model

    live_policy = make_check_expense_policy(
        init_chat_model(
            "anthropic:claude-sonnet-4-6", api_key=get_settings().anthropic_api_key, max_tokens=256
        )
    )
    print("live    lodging  268.40", live_policy("lodging", 268.40))
else:
    print("\n(no key -- showing the scripted judge; a live Sonnet reads the prose in class.)")

meals       12.75 {"within_policy": true, "citation": "Individual meals while travelling are reimbursed up to $50 per person.", "reason": "12.75 is within the $50 meals limit."}
lodging     268.4 {"within_policy": false, "citation": "Hotels are reimbursed up to $250 per night, taxes included.", "reason": "268.40 is over the $250 lodging limit \u2014 flag for approval."}

(no key -- showing the scripted judge; a live Sonnet reads the prose in class.)


Register all three — the one you wrote (`find_matching_record`), the reused `calculator`, and the provided `check_expense_policy` — so the agent has a *real* three-tool decision to make: match, total, or check the policy. That decision is what makes its trace worth reading. **The authoring budget is still one tool.** If you find yourself *writing* a second, that's your own project (Section 6), not this slice.

In [7]:
# name -> function: the exact shape agent_graph binds and dispatches (like common.tools.TOOLS).
L50_TOOLS = {
    "find_matching_record": find_matching_record,
    "calculator": calculator,
    "check_expense_policy": check_expense_policy,
}
print("tools:", list(L50_TOOLS))

tools: ['find_matching_record', 'calculator', 'check_expense_policy']


[↑ Back to top](#top)

## 3. Assemble the shallow agent

In [L11](../L11/L1101_intro.md) you met the shallow agent as one line — `create_agent(model, tools)` — the L10 model→tool→model loop, packaged. From [L12](../L12/L1201_intro.md) on, the mini-arc runs that same ReAct loop through **`agent_graph.run` / `arun`**, which hands the run back as a `RunResult` carrying a **`.trace`** — the observability seam that makes Sections 4 and 5 possible. Same loop; now you can *see* it. That's the one stable call the rest of the walkthrough uses.

To keep the walkthrough reproducible, we drive it with a **scripted `FakeModel`** — it plays a fixed sequence of tool calls, but the *tools really run* and the *trace is real*. (In Section 4 you'll swap in live Sonnet with one gated cell.)

In [8]:
coffee = receipt_text("R-coffee")

# A scripted run of the happy path: find the record, total the line items to confirm it
# reconciles, then check the expense policy -- three tools, a real decision at each step.
# The FakeModel supplies the decisions; agent_graph runs the real tools and captures the real trace.
good = await arun(
    FakeModel(
        [
            tool_reply(tool_call("m1", "find_matching_record", {"receipt": coffee})),
            tool_reply(tool_call("c1", "calculator", {"expression": "5.50 + 3.25 + 4.00"})),
            tool_reply(
                tool_call("p1", "check_expense_policy", {"category": "meals", "amount": 12.75})
            ),
            text_reply(
                "Matched EXP-1001 (Blue Bottle Coffee, $12.75); the items total $12.75 so it "
                "reconciles, and it's within the $50 meals guidance."
            ),
        ]
    ),
    L50_TOOLS,
    f"Reconcile this receipt against the ledger:\n{coffee}",
)

print("final:", good.final_text)
print("termination:", good.termination, "| iterations:", good.iterations)
print("tools used:", tool_trajectory(good))

final: Matched EXP-1001 (Blue Bottle Coffee, $12.75); the items total $12.75 so it reconciles, and it's within the $50 meals guidance.
termination: natural | iterations: 4
tools used: ['find_matching_record', 'calculator', 'check_expense_policy']


It terminated `natural` — the model stopped because it was done, not because it hit the cap. That step cap (`max_steps`, default 8) is a **design choice for this task**, not a magic default: reconciling one receipt is at most a few tool calls, so a low cap is a cheap guardrail against a runaway — which is exactly what you'll see next.

[↑ Back to top](#top)

## 4. Trace it and find a real failure

Now the L12 skill, on *your* agent: read the trace to see what actually happened, and locate a failure **from the trace alone** — not by guessing. The failure here isn't one the course planted; it's on your task. Feed the agent the **malformed** receipt (OCR failed — no readable vendor, an unparseable amount). A naive first cut keeps calling the matcher on the same bad input and never stops:

In [9]:
mystery = receipt_text("R-mystery")

# The buggy behavior: the agent retries find_matching_record on the SAME unreadable receipt
# instead of accepting the "no confident match" and stopping. A low cap forces the halt.
buggy = await arun(
    FakeModel(
        [
            tool_reply(tool_call(f"b{i}", "find_matching_record", {"receipt": mystery}))
            for i in range(6)
        ]
    ),
    L50_TOOLS,
    f"Reconcile this receipt:\n{mystery}",
    max_steps=3,
)

narrate(buggy)
print("\ntermination:", buggy.termination, "| final text:", repr(buggy.final_text))

[0] chain agent_graph.run      {'termination': 'max_steps', 'iterations': 3}
[1] llm   agent                -> tool_calls=['find_matching_record'] tokens=120
[2] tool  find_matching_record {'receipt': '{"note": "scanned blob -- OCR failed", "amt": "??", "lines": ["illegible"]}'} -> '{"match": null, "reason": "unrecognized receipt format"}'
[3] llm   agent                -> tool_calls=['find_matching_record'] tokens=120
[4] tool  find_matching_record {'receipt': '{"note": "scanned blob -- OCR failed", "amt": "??", "lines": ["illegible"]}'} -> '{"match": null, "reason": "unrecognized receipt format"}'
[5] llm   agent                -> tool_calls=['find_matching_record'] tokens=120
[6] tool  find_matching_record {'receipt': '{"note": "scanned blob -- OCR failed", "amt": "??", "lines": ["illegible"]}'} -> '{"match": null, "reason": "unrecognized receipt format"}'

termination: max_steps | final text: ''


Read the trace, don't theorize: the **same `(name, args)` pair repeats** and the run ended on `max_steps`, never `natural`. In L12's vocabulary that's a **runaway** — and the tool result was `"match": null` *every time*, so the agent had the answer on turn one and ignored it. That's the failure. **Now see it in the real tool** — one gated cell pushes a live Sonnet run to the cohort's Langfuse, where the same run reads as a waterfall:

In [10]:
if LIVE_LLM and LIVE_LANGFUSE:
    from langchain.chat_models import init_chat_model
    from langfuse import Langfuse
    from langfuse.langchain import CallbackHandler

    from fluffy_potato_curriculum.common.config import require_langfuse

    host, public_key, secret_key = require_langfuse()
    client = Langfuse(host=host, public_key=public_key, secret_key=secret_key)

    # A pre-made trace id links the callback's spans to the URL we print.
    trace_id = client.create_trace_id()
    handler = CallbackHandler(trace_context={"trace_id": trace_id})
    model = init_chat_model(
        "anthropic:claude-sonnet-4-6", api_key=get_settings().anthropic_api_key
    ).with_config(callbacks=[handler])

    live = await arun(model, L50_TOOLS, f"Reconcile this receipt against the ledger:\n{coffee}")
    client.flush()
    print("live final:", live.final_text)
    print("open this run in Langfuse:", client.get_trace_url(trace_id=trace_id))
else:
    print(
        "No live Sonnet + Langfuse -- running on the scripted FakeModel spine (reproducible offline)."
    )
    print(
        "In class, set ANTHROPIC_API_KEY + the Langfuse keys to push this run to the cohort dashboard,"
    )
    print(
        "then read the trace there: same spans, GENERATION for the model calls, SPAN for the tools."
    )

No live Sonnet + Langfuse -- running on the scripted FakeModel spine (reproducible offline).
In class, set ANTHROPIC_API_KEY + the Langfuse keys to push this run to the cohort dashboard,
then read the trace there: same spans, GENERATION for the model calls, SPAN for the tools.


Whether you read it inline (`narrate`) or in Langfuse's waterfall, it's the same trace and the same finding. **Trace before you guess** — the trace says what happened; without it you'd be theorizing about a non-deterministic run.

[↑ Back to top](#top)

## 5. Turn the failure into one eval case

Here's the habit the whole mini course was building toward: **a failure becomes a kept case.** You just watched the runaway in the trace — now write the [L13](../L13/L1301_intro.md) loop's payoff as code: *trace a failure → write a case that catches it → keep it forever.* This is the minimal close — **one case, one scorer, one run**, pure Python. (Scaling it to a Langfuse Dataset/Experiment with a pass rate and a Sonnet-vs-Haiku A/B is your **bonus** below — you already learned that machinery in L13.)

A good scorer **fails when the bug is present and passes when it's fixed**. The bug is a runaway, so the scorer reads the *trajectory* (`run.trace`) and fails on a repeated identical call:

In [11]:
def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: fail if the agent repeats the exact same tool call (a runaway)."""
    calls = tool_calls(run)
    signatures = {(name, tuple(sorted(args.items()))) for name, args in calls}
    passed = len(signatures) == len(calls) and run.termination == "natural"
    return EvalResult(
        key="no_runaway", score=passed, comment=f"{len(calls)} calls, {run.termination}"
    )


def flags_no_match(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer: fail unless the final answer actually reports the no-match, not a guess."""
    text = run.final_text.lower()
    passed = any(p in text for p in ("no confident match", "no match", "couldn't", "manual review"))
    return EvalResult(key="flags_no_match", score=passed, comment=run.final_text[:48])


case = EvalCase(id="malformed_receipt", inputs={"task": mystery})

Now the fix: a good agent calls the matcher **once**, reads the `"match": null`, and reports it plainly. Run the one case against the buggy run and the fixed run — the scorer must go **False → True**:

In [12]:
# The FIXED behavior: one call, accept the no-match, stop.
fixed = await arun(
    FakeModel(
        [
            tool_reply(tool_call("f1", "find_matching_record", {"receipt": mystery})),
            text_reply(
                "I couldn't find a confident match for that receipt, so I've flagged it for manual review."
            ),
        ]
    ),
    L50_TOOLS,
    f"Reconcile this receipt:\n{mystery}",
    max_steps=3,
)

for label, run in [("buggy", buggy), ("fixed", fixed)]:
    verdicts = [no_runaway(run=run, example=case), flags_no_match(run=run, example=case)]
    line = "  ".join(f"{v.key}={v.passed}" for v in verdicts)
    print(f"{label:6} -> {line}")

buggy  -> no_runaway=False  flags_no_match=False
fixed  -> no_runaway=True  flags_no_match=True


That's the ratchet: **the slice isn't done when it runs — it's done when it's traced and has one case that would catch its regression.** That case now fails the day someone reintroduces the runaway, forever.

> **Bonus (not demoed — try it on your own agent).** You already learned the platform half of L13: `upload_dataset` these cases as a Langfuse **Dataset**, run them as an **Experiment** with K samples to get a pass *rate*, and re-run against **Haiku 4.5** to compare Sonnet-vs-Haiku in the run-comparison view. That's the on-ramp to the end-of-week project — where a real eval set earns its keep.

[↑ Back to top](#top)

## 6. Reflect and hand off

Scroll back up. You went from a blank cell to a **traced, evaluated shallow agent** — and here's where each of the five mini-cut objectives showed up on *this* concrete build:

- **Design a tool (L07/L08)** → Section 2, `find_matching_record` — the one tool you wrote (the agent also runs the provided `check_expense_policy`, an LLM read of the policy prose).
- **Shallow agent (L10/L11)** → Section 3, `agent_graph.run` — the loop, packaged, choosing among three tools.
- **Trace & read it (L12)** → Section 4, `narrate(run)` and the Langfuse push.
- **Evaluate it (L13)** → Section 5, one case + two scorers, False → True.
- **Scope & finish (the capstone skill)** → Sections 1 and the fact that you *stopped*.

The **first thing you'd add next** is the seam into your own project. The canonical one here is **reimbursement detection** — the stretch tool that was deliberately *not* in the core slice:

In [13]:
# The stretch tool, imported from the tested sibling -- this is "what you'd add next", not today's build.
from fluffy_potato_curriculum.lessons.L50.receipt_tools import check_reimbursement

# The hotel expense was paid back by a bank credit; the coffee was not.
for expense_id in ("EXP-1003", "EXP-1001"):
    print(f"{expense_id}:", check_reimbursement(expense_id))

EXP-1003: {"reimbursed": true, "transaction": {"txn_id": "BANK-501", "date": "2026-06-10", "description": "REIMBURSEMENT - HARBOR GRAND HOTEL", "amount": 268.4, "type": "credit"}}
EXP-1001: {"reimbursed": false, "expense_id": "EXP-1001"}


That's a *second* tool, a *harder* task, a *tighter* scorer — the shape of the independent [end-of-week project](../../../../docs/origin/PROJECT_BRIEF_DESIGN.md), which is the same motion as today, wider and yours. Everything you used is in `common/` — you now know how little new code a real shallow agent actually needs.

**Next read:** [MINI_WRAPUP.md](../MINI_WRAPUP.md) — the course-level retrospective this lesson deliberately doesn't try to be.

[↑ Back to top](#top)